<a href="https://colab.research.google.com/github/algroznykh/closed_form_nca/blob/main/closed_form_spectral_featureviz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# -*- coding: utf-8 -*-
"""
Decentralized Closed-Form NCA operating directly in the Frequency Domain.
Propagates waves from a localized DC seed outward to high-frequency spectral bands.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import cv2
import io
import numpy as np
import traceback
import ipywidgets as widgets
from IPython.display import display, HTML
from collections import deque
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Colab-specific interface module
from google.colab import output

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- SECTION 1: CLIP INITIALIZATION & ACTIVATION HOOKS ---
try:
    import clip
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "git+https://github.com/openai/CLIP.git"])
    import clip

clip_model, _ = clip.load('RN101', device=device, jit=False)
clip_model.eval().float()
for p in clip_model.parameters():
    p.requires_grad_(False)

CLIP_MEAN = torch.tensor([0.4814, 0.4578, 0.4082], device=device).view(1, 3, 1, 1)
CLIP_STD  = torch.tensor([0.2686, 0.2613, 0.2758], device=device).view(1, 3, 1, 1)

def clip_norm(x):
    return (x - CLIP_MEAN) / CLIP_STD

_act = {}
def get_hook(name):
    def _hook(mod, inp, out):
        _act[name] = out
    return _hook

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    dict(clip_model.visual.named_modules())[layer_name].register_forward_hook(get_hook(layer_name))


# --- SECTION 2: MODEL COMPONENTS ---

class ComplexLift(nn.Module):
    """
    Strictly 1x1 point-wise complex lifting layer with zero additive bias.
    Maps real/imaginary inputs to the high-dimensional complex latent space.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        # Robust initialization to ensure seed signal propagates into latents
        nn.init.normal_(self.conv.weight, std=0.2)

    def forward(self, x):
        return torch.complex(self.conv(x.real), self.conv(x.imag))


class PointwiseSpectralProjector(nn.Module):
    """
    Strictly 1x1 point-wise projection head.
    Maps complex spatial representation to RGB.
    """
    def __init__(self, latent_ch, out_ch=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(latent_ch * 2, latent_ch, kernel_size=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(latent_ch, out_ch, kernel_size=1, bias=False)
        )
        # High contrast weight initialization
        for m in self.net.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.15)

    def forward(self, x):
        x_stacked = torch.cat([x.real, x.imag], dim=1)
        raw = self.net(x_stacked)
        return torch.clamp(raw + 0.5, 0.0, 1.0)


class ClosedFormNCA(nn.Module):
    """
    Spectral Cellular Automaton running directly in the frequency domain.
    Evolves wave structures from DC to high frequencies.
    """
    def __init__(self, in_ch=12, latent_ch=48, kernel_size=3):
        super().__init__()
        self.C = latent_ch
        self.k_size = kernel_size
        self.shift = kernel_size // 2

        self.lift = ComplexLift(in_ch, latent_ch)
        self.project = PointwiseSpectralProjector(latent_ch, out_ch=3)

        # Complex Unitary Channel Mixing parameters (skew-Hermitian generator)
        self.R = nn.Parameter(torch.randn(latent_ch, latent_ch) * 0.05)
        self.I = nn.Parameter(torch.randn(latent_ch, latent_ch) * 0.05)

        # Learnable anisotropic propagation kernel acting on the frequency grid
        self.spatial_kernel = nn.Parameter(torch.randn(latent_ch, 1, kernel_size, kernel_size) * 0.02)

        # Initialize spatial kernel with self-amplifying bias to grow by default
        with torch.no_grad():
            for c in range(latent_ch):
                # Strong center positive value for default growth
                self.spatial_kernel.data[c, 0, 1, 1] = 0.25
                # Small random anisotropic values to break symmetry early
                self.spatial_kernel.data[c, 0, 0, :] += torch.randn(3) * 0.02
                self.spatial_kernel.data[c, 0, 2, :] += torch.randn(3) * 0.02

        # Initialize viscosity raw low (around -3.0 -> Softplus(-3.0) approx 0.048)
        self.viscosity_raw = nn.Parameter(torch.randn(latent_ch, 1, 1) * 0.1 - 3.0)

        # Stable isotropic discrete Laplacian kernel for physical viscosity damping
        lap_k = torch.tensor([[1.0,  2.0, 1.0],
                              [2.0, -12.0, 2.0],
                              [1.0,  2.0, 1.0]], dtype=torch.float32) / 12.0
        self.register_buffer('laplacian_kernel', lap_k.view(1, 1, 3, 3).repeat(latent_ch, 1, 1, 1))

        # Pre-compute distance-squared spectral damping grid for 128x128 resolution
        # Raised to power of 3 (representing |k|^6) to act as an absolute spectral wall at borders
        Y, X = torch.meshgrid(
            torch.linspace(-1.0, 1.0, 128),
            torch.linspace(-1.0, 1.0, 128),
            indexing='ij'
        )
        self.register_buffer('spectral_damping', ((X**2 + Y**2) ** 3).view(1, 1, 128, 128))

    def get_unitary_matrix(self):
        """Generates a strictly unitary matrix conserving L2 channel energy."""
        A = torch.complex(self.R, self.I)
        A = torch.clamp(A.real, -5.0, 5.0) + 1j * torch.clamp(A.imag, -5.0, 5.0)
        H = A - A.adjoint()
        return torch.linalg.matrix_exp(H)

    def get_dynamic_damping(self, H, W, device):
        """Generates resolution-agnostic spectral viscosity walls (|k|^6)."""
        if H == 128 and W == 128:
            return self.spectral_damping
        Y, X = torch.meshgrid(
            torch.linspace(-1.0, 1.0, H, device=device),
            torch.linspace(-1.0, 1.0, W, device=device),
            indexing='ij'
        )
        return ((X**2 + Y**2) ** 3).view(1, 1, H, W)

    def forward(self, t, x0_fft):
        """Analytical closed-form transition solved via spatial-spectral dual operators."""
        with torch.amp.autocast('cuda', enabled=False):
            t = t.to(torch.float32)
            x0_fft = x0_fft.to(torch.complex64)

            # Self-limiting logistic time-saturation to guarantee long-term analytical stability
            alpha = 0.05
            t_saturated = (1.0 - torch.exp(-alpha * t)) / alpha

            # Lift inputs to high-dimensional latent space
            z0 = self.lift(x0_fft)
            B, C, H, W = z0.shape

            U = self.get_unitary_matrix()
            viscosity = F.softplus(self.viscosity_raw)
            damping_mask = self.get_dynamic_damping(H, W, x0_fft.device)

            # 1. Spatial Dual representation of the unconstrained spectral convolution kernel
            pad_h, pad_w = H - self.k_size, W - self.k_size
            k_padded = F.pad(self.spatial_kernel, (0, pad_w, 0, pad_h))
            k_padded = torch.roll(k_padded, (-self.shift, -self.shift), dims=(2, 3))

            # Scale by (H * W) to cancel the IFFT normalization and align with spectral convolutions
            k_spatial = torch.fft.ifft2(k_padded).squeeze(1) * (H * W)

            # 2. Linear propagation in dual spatial space
            # Rotate initial spectral state and transform to spatial domain
            z0_rot_fft = torch.einsum('cd, bdhw -> bchw', U.adjoint(), z0)
            z0_spatial = torch.fft.ifft2(torch.fft.ifftshift(z0_rot_fft))

            # Multiply by spatial dual growth exponent
            z_t_spatial = z0_spatial * torch.exp(t_saturated * k_spatial.unsqueeze(0))

            # 3. Apply viscosity damping back in the frequency domain
            Z_t_spatial_fft = torch.fft.fftshift(torch.fft.fft2(z_t_spatial))
            Z_t_spatial_fft = Z_t_spatial_fft * torch.exp(-viscosity * damping_mask * t_saturated)

            # 4. Map back to spatial domain and rotate to channel coordinates
            z_t_spatial = torch.fft.ifft2(torch.fft.ifftshift(Z_t_spatial_fft))
            z_t = torch.einsum('cd, bdhw -> bchw', U, z_t_spatial)

            # 5. Point-wise local complex magnitude-clamping activation
            mag = torch.sqrt(z_t.real**2 + z_t.imag**2 + 1e-8)
            scale = torch.tanh(mag) / (mag + 1e-8)
            z_final = z_t * scale

            rgb = self.project(z_final)

        return z_final, rgb

    def step_latent_euler(self, z_fft, dt=0.05):
        """Strictly local step-by-step convolutions executed directly on the frequency grid."""
        U = self.get_unitary_matrix()
        viscosity = F.softplus(self.viscosity_raw)

        B, C, H, W = z_fft.shape
        damping_mask = self.get_dynamic_damping(H, W, z_fft.device)

        # 1. Rotate to energy-conserving diagonalized channel space
        w_fft = torch.einsum('cd, bdhw -> bchw', U.adjoint(), z_fft)

        # 2. Convolve directly on the frequency grid (spectral propagation)
        padded_w = F.pad(w_fft.real, (1, 1, 1, 1)) + 1j * F.pad(w_fft.imag, (1, 1, 1, 1))
        conv_r = F.conv2d(padded_w.real, self.spatial_kernel, groups=self.C)
        conv_i = F.conv2d(padded_w.imag, self.spatial_kernel, groups=self.C)
        dw_fft_conv = torch.complex(conv_r, conv_i)

        # 3. Apply isotropic spectral viscosity damping (suppresses high-frequency boundaries)
        dw_fft = dw_fft_conv - viscosity * damping_mask * w_fft

        # 4. Rotate back to channel coordinates and perform Euler integration step
        dz_fft = torch.einsum('cd, bdhw -> bchw', U, dw_fft)
        z_new_fft = z_fft + dt * dz_fft

        # 5. Apply pointwise complex magnitude clamping in spatial domain to preserve physical boundaries
        z_spatial = torch.fft.ifft2(torch.fft.ifftshift(z_new_fft))
        mag = torch.sqrt(z_spatial.real**2 + z_spatial.imag**2 + 1e-8)
        scale = torch.tanh(mag) / (mag + 1e-8)
        z_spatial_saturated = z_spatial * scale

        # 6. Transform back to frequency domain state
        return torch.fft.fftshift(torch.fft.fft2(z_spatial_saturated))


# --- SECTION 3: TASK DEFINITION ---

class CLIPFeatureTargetTask:
    """CLIP optimization task using local spectral initialization (DC Seed)."""
    def generate_input(self, batch_size, channels, size, device):
        # Create an empty spectral grid
        x_fft = torch.zeros(batch_size, channels, size, size, dtype=torch.complex64, device=device)

        # Populate only the central DC and low-frequency components (3x3 region) with robust energy
        r = 1
        mid = size // 2

        real_noise = torch.randn(batch_size, channels, 2*r+1, 2*r+1, device=device) * 0.5
        imag_noise = torch.randn(batch_size, channels, 2*r+1, 2*r+1, device=device) * 0.5

        x_fft[:, :, mid-r:mid+r+1, mid-r:mid+r+1] = torch.complex(real_noise, imag_noise)
        return x_fft

    def _random_crops(self, img, n=2, size=224):
        B, C, H, W = img.shape
        scales = torch.empty(n, device=img.device).uniform_(0.7, 1.2)
        tx = torch.empty(n, device=img.device).uniform_(-0.1, 0.1)
        ty = torch.empty(n, device=img.device).uniform_(-0.1, 0.1)
        flip = (torch.rand(n, device=img.device) < 0.5).float() * 2 - 1

        theta = torch.zeros(n, 2, 3, device=img.device)
        theta[:, 0, 0] = scales * flip
        theta[:, 1, 1] = scales
        theta[:, 0, 2] = tx
        theta[:, 1, 2] = ty

        grid = F.affine_grid(theta, (n, C, size, size), align_corners=False)
        expanded = img.unsqueeze(1).expand(-1, n, -1, -1, -1).reshape(B * n, C, H, W)
        return F.grid_sample(expanded, grid.repeat(B, 1, 1, 1), padding_mode='reflection', align_corners=False)

    def compute_loss(self, rgb, config):
        views = self._random_crops(rgb, n=2)
        views = views + torch.randn_like(views) * 0.01

        _ = clip_model.encode_image(clip_norm(views))
        current_layer = config.get("target_layer", "layer2")
        current_acts = _act[current_layer]

        if config.get("is_deep_dream", False):
            fv_loss = -current_acts.square().mean() * 5.0
        else:
            target_ch = config.get("target_channel", 42)
            fv_loss = -current_acts[:, target_ch].mean() * 5.0

        l2_reg = rgb.square().mean() * 0.1
        tv_loss = ((rgb[:, :, :, :-1] - rgb[:, :, :, 1:]).square().mean() +
                   (rgb[:, :, :-1, :] - rgb[:, :, 1:, :]).square().mean()) * 0.005

        Z_loss = torch.fft.rfft2(rgb.float())
        mag = torch.abs(Z_loss)
        mag_ac = mag.clone()
        mag_ac[:, :, 0, 0] = 0.0

        max_ac = mag_ac.amax(dim=(2, 3))
        mean_ac = mag_ac.mean(dim=(2, 3))
        ratio = max_ac / (mean_ac + 1e-6)
        spec_consolidation = - torch.log(ratio + 1.0).mean() * 0.1

        total_loss = fv_loss + l2_reg + tv_loss + spec_consolidation
        return total_loss, {
            "total": total_loss.item(),
            "feature": fv_loss.item(),
            "l2": l2_reg.item(),
            "tv": tv_loss.item(),
            "sparsity": spec_consolidation.item()
        }


# --- SECTION 4: TRAINING REGIME ---

class NCATrainer:
    def __init__(self, model, task, lr=4e-3, weight_decay=1e-4):
        self.model = model
        self.task = task
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scaler = torch.amp.GradScaler('cuda')

        self.logs = {"total": [], "feature": [], "l2": [], "tv": [], "sparsity": []}
        self.best_loss = float('inf')
        self.train_times = deque(maxlen=30)

    def step(self, config):
        t_start = time.time()
        t_train = torch.empty(2, 1, 1, 1, device=device).uniform_(10.0, 40.0)
        x0 = self.task.generate_input(batch_size=2, channels=12, size=128, device=device)

        with torch.amp.autocast('cuda'):
            _, rgb = self.model(t_train, x0)
            loss, loss_breakdown = self.task.compute_loss(rgb, config)

        self.optimizer.zero_grad()
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.optimizer)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()

        for k, v in loss_breakdown.items():
            if k in self.logs:
                self.logs[k].append(v)

        if loss_breakdown["feature"] < self.best_loss:
            self.best_loss = loss_breakdown["feature"]

        self.train_times.append(time.time() - t_start)
        return loss_breakdown

    def reset_logs(self):
        for k in self.logs:
            self.logs[k].clear()
        self.best_loss = float('inf')


# --- SECTION 5: MODULAR UI & CONTROLLER SYSTEM ---

class Dashboard:
    def __init__(self, model_class, task_class, trainer_class):
        self.model_class = model_class
        self.task_class = task_class
        self.trainer_class = trainer_class

        self.init_system()

        # Build controls
        self.layer_dropdown = widgets.Dropdown(
            options=['layer1', 'layer2', 'layer3', 'layer4'],
            value='layer2', description='Layer:', layout=widgets.Layout(width='180px')
        )
        self.channel_selector = widgets.BoundedIntText(
            min=0, max=511, value=42, description='Channel:', layout=widgets.Layout(width='160px')
        )
        self.dd_toggle = widgets.ToggleButton(
            value=False, description='DeepDream Mode', button_style='', icon='cloud', layout=widgets.Layout(width='160px')
        )
        self.btn_hard_reset = widgets.Button(
            description='💣 Hard Reset Model', button_style='danger', icon='bomb', layout=widgets.Layout(width='180px')
        )
        self.zoom_slider = widgets.IntSlider(
            min=384, max=1536, value=768, step=96, description='Scale Viewport:', layout=widgets.Layout(width='300px')
        )
        self.btn_record = widgets.ToggleButton(
            description='🔴 Record Video', button_style='info', icon='video-camera', layout=widgets.Layout(width='150px')
        )

        # Outputs
        self.img_widget = widgets.Image(
            value=cv2.imencode('.jpg', np.zeros((128, 384, 3), np.uint8))[1].tobytes(),
            format='jpeg'
        )
        self.img_widget.add_class("colab-nca-viewport")
        self.img_widget.layout.width = "768px"
        self.img_widget.layout.height = "256px"

        self.fps_label = widgets.Label(value='Ready')
        self.stats_label = widgets.HTML(value="Waiting for metrics...", layout=widgets.Layout(margin='0 0 0 15px'))
        self.graph_widget = widgets.Image(format='png', width=500, height=200)
        self._update_graph_placeholder("Waiting for loss data...")

        self.is_running = True
        self.is_paused = False
        self.t = 0.0
        self._pause_offset = 0.0
        self._unpause_wall = time.time()
        self.recorded_frames = []

        self._setup_handlers()

    def init_system(self):
        self.model = self.model_class(in_ch=12, latent_ch=48, kernel_size=3).to(device)
        self.task = self.task_class()
        self.trainer = self.trainer_class(self.model, self.task)

        self.display_x0 = self.task.generate_input(batch_size=1, channels=12, size=128, device=device)
        self.z_rk4 = self.model.lift(self.display_x0)

    def _setup_handlers(self):
        self.layer_dropdown.observe(self._on_layer_change, 'value')
        self.channel_selector.observe(self._on_loss_param_change, 'value')
        self.dd_toggle.observe(self._on_dd_toggle, 'value')
        self.btn_hard_reset.on_click(self._on_hard_reset_click)
        self.zoom_slider.observe(self._on_zoom_change, 'value')
        self.btn_record.observe(self._on_record_toggle, 'value')

        self.btn_toggle = widgets.ToggleButton(value=True, icon='stop', button_style='danger', layout=widgets.Layout(width='32px'))
        self.btn_toggle.observe(self._on_play_stop_toggle, names='value')
        self.btn_pause = widgets.ToggleButton(icon='pause', button_style='warning', layout=widgets.Layout(width='32px'))
        self.btn_pause.observe(self._on_pause_toggle, names='value')
        self.btn_reset = widgets.Button(icon='refresh', button_style='info', layout=widgets.Layout(width='32px'))
        self.btn_reset.on_click(lambda _: self.reset_time())

    def _on_layer_change(self, change):
        ch_map = {'layer1': 256, 'layer2': 512, 'layer3': 1024, 'layer4': 2048}
        self.channel_selector.max = ch_map[change['new']] - 1
        self.trainer.reset_logs()

    def _on_loss_param_change(self, change):
        self.trainer.reset_logs()

    def _on_dd_toggle(self, change):
        self.channel_selector.disabled = change['new']
        self.dd_toggle.button_style = 'success' if change['new'] else ''
        self.trainer.reset_logs()

    def _on_zoom_change(self, change):
        new_w = change['new']
        self.img_widget.layout.width = f"{new_w}px"
        self.img_widget.layout.height = f"{new_w // 3}px"

    def _on_record_toggle(self, change):
        if change['new']:
            self.recorded_frames = []
            self.btn_record.description = '⏹️ Stop & Download'
            self.btn_record.button_style = 'danger'
        else:
            self.btn_record.description = '⏳ Processing...'
            self.btn_record.disabled = True
            if self.recorded_frames:
                try:
                    h, w, c = self.recorded_frames[0].shape
                    filename = 'nca_simulation.mp4'
                    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                    out_video = cv2.VideoWriter(filename, fourcc, 30.0, (w, h))
                    for frame in self.recorded_frames:
                        out_video.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
                    out_video.release()

                    from google.colab import files
                    files.download(filename)
                    self.stats_label.value = "💾 Recording exported successfully."
                except Exception as ev:
                    print("Video recording encoding error:", ev)
            self.btn_record.description = '🔴 Record Video'
            self.btn_record.button_style = 'info'
            self.btn_record.disabled = False
            self.recorded_frames = []

    def _on_play_stop_toggle(self, change):
        self.is_running = change['new']
        if self.is_running:
            self.btn_toggle.icon = 'stop'
            self.btn_toggle.button_style = 'danger'
            self._unpause_wall = time.time()
        else:
            self.btn_toggle.icon = 'play'
            self.btn_toggle.button_style = 'success'

    def _on_pause_toggle(self, change):
        self.is_paused = change['new']
        if self.is_paused:
            self._pause_offset = self.t
            self.btn_pause.icon = 'play'
        else:
            self._unpause_wall = time.time()
            self.btn_pause.icon = 'pause'

    def _on_hard_reset_click(self, change):
        self.init_system()
        self._update_graph_placeholder("Universe Destroyed. Rebuilding...")
        self.stats_label.value = "⚡ Physics Re-rolled | 🏆 Loss: inf"

    def reset_time(self):
        self.t = 0.0
        self._pause_offset = 0.0
        self._unpause_wall = time.time()
        self.display_x0 = self.task.generate_input(batch_size=1, channels=12, size=128, device=device)
        self.z_rk4 = self.model.lift(self.display_x0)

    def _update_graph_placeholder(self, text):
        fig, ax = plt.subplots(figsize=(6, 2.5))
        ax.text(0.5, 0.5, text, ha='center', va='center', color='gray')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        fig.tight_layout(pad=1.0)
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        plt.close(fig)
        self.graph_widget.value = buf.getvalue()

    def get_current_config(self):
        return {
            "target_layer": self.layer_dropdown.value,
            "target_channel": self.channel_selector.value,
            "is_deep_dream": self.dd_toggle.value
        }

    def render_panel(self):
        legend_label = widgets.HTML(
            value=f"<div style='display:flex; width:100%; text-align:center; font-family:sans-serif; font-size:12px; font-weight:bold; color:#aaa; background:#1e1e1e; padding:6px 0; border-bottom:1px solid #333;'><div style='width:33.33%; border-right:1px solid #333;'>⚡ Closed-Form O(1) Stable State</div><div style='width:33.33%; border-right:1px solid #333;'>🦠 Spectral Euler CA (FFT Grid)</div><div style='width:33.33%;'>🔮 2D FFT Spectrum [R=Mag, G=Phase]</div></div>"
        )

        viewport_ui = widgets.VBox([
            legend_label,
            self.img_widget,
            widgets.HBox([self.btn_toggle, self.btn_pause, self.btn_reset, self.zoom_slider, self.btn_record, self.fps_label],
                         layout=widgets.Layout(align_items='center', margin='5px 0'))
        ], layout=widgets.Layout(border='1px solid #333', padding='10px', border_radius='4px', background_color='#1a1a1a'))

        controls_styled = widgets.HBox([
            self.layer_dropdown, self.channel_selector, self.dd_toggle, self.btn_hard_reset
        ], layout=widgets.Layout(border='1px solid #333', padding='10px', margin='5px 0', border_radius='4px', background_color='#1a1a1a', align_items='center'))

        layout_block = widgets.VBox([
            viewport_ui,
            controls_styled,
            widgets.HBox([self.graph_widget, self.stats_label],
                         layout=widgets.Layout(border='1px solid #333', padding='10px', border_radius='4px', background_color='#1a1a1a', align_items='center'))
        ], layout=widgets.Layout(padding='10px', background_color='#111', border_radius='8px'))

        display(layout_block)


# --- SECTION 6: RUNTIME PIPELINE ---

dashboard = Dashboard(
    model_class=ClosedFormNCA,
    task_class=CLIPFeatureTargetTask,
    trainer_class=NCATrainer
)
dashboard.render_panel()

last_render_time = 0.0
last_graph_update = 0.0
last_ui_update = 0.0
render_frames_history = deque(maxlen=30)


def colab_render_step():
    global last_render_time
    try:
        if not dashboard.is_running:
            return

        t_now = time.time()
        if not dashboard.is_paused:
            new_t = dashboard._pause_offset + (t_now - dashboard._unpause_wall) * 2.0
            dt = new_t - dashboard.t
            dashboard.t = new_t
        else:
            dt = 0.0

        with torch.no_grad():
            t_tensor = torch.tensor([dashboard.t], device=device).view(1, 1, 1, 1)
            _, rgb_cf = dashboard.model(t_tensor, dashboard.display_x0)

            # High-speed Spectral Euler integration convolving directly on the FFT Grid
            sub_steps = max(1, int(dt / 0.1))
            for _ in range(sub_steps):
                dashboard.z_rk4 = dashboard.model.step_latent_euler(dashboard.z_rk4, dt=dt/sub_steps)

            # Get spatial representations to project to RGB
            z_rk4_spatial = torch.fft.ifft2(torch.fft.ifftshift(dashboard.z_rk4))

            rgb_cf_proj = rgb_cf[0].permute(1, 2, 0)
            rgb_rk4_proj = dashboard.model.project(z_rk4_spatial)[0].permute(1, 2, 0)

            # Compute local 2D FFT Spectrum of the active Spectral Euler visualization
            gray = 0.2989 * rgb_rk4_proj[:, :, 0] + 0.5870 * rgb_rk4_proj[:, :, 1] + 0.1140 * rgb_rk4_proj[:, :, 2]
            Y = torch.fft.fft2(gray.float())
            Y_shifted = torch.fft.fftshift(Y)

            # Map Magnitude to Red channel
            mag = torch.abs(Y_shifted)
            log_mag = torch.log1p(mag)
            log_mag_norm = (log_mag - log_mag.min()) / (log_mag.max() - log_mag.min() + 1e-5)

            # Map Phase to Green/Blue channels, weighted by log magnitude to eliminate high-frequency green noise
            phase = torch.angle(Y_shifted)
            phase_norm = (phase + np.pi) / (2.0 * np.pi)

            fft_view = torch.zeros_like(rgb_rk4_proj)
            fft_view[:, :, 0] = log_mag_norm                              # Red: Magnitude
            fft_view[:, :, 1] = phase_norm * log_mag_norm                 # Green: Phase (magnitude-masked)
            fft_view[:, :, 2] = (1.0 - phase_norm) * log_mag_norm         # Blue: Inverse Phase (magnitude-masked)

            combined_spatial = torch.cat([rgb_cf_proj, rgb_rk4_proj, fft_view], dim=1)
            img_np = (combined_spatial.clamp(0, 1).mul_(255)).byte().cpu().numpy()

        dashboard.img_widget.value = cv2.imencode('.jpg', img_np[:, :, ::-1], [int(cv2.IMWRITE_JPEG_QUALITY), 75])[1].tobytes()

        if dashboard.btn_record.value:
            dashboard.recorded_frames.append(img_np.copy())
            if len(dashboard.recorded_frames) >= 900:
                dashboard.btn_record.value = False

        render_frames_history.append(t_now - last_render_time)
        dashboard.fps_label.value = f"FPS: {1.0/max(sum(render_frames_history)/len(render_frames_history), 1e-4):.0f} | t={dashboard.t:.2f}"
        last_render_time = t_now
    except Exception:
        print("\nRENDER ERROR:\n", traceback.format_exc())


def colab_train_step():
    global last_graph_update, last_ui_update
    try:
        if not dashboard.is_running or dashboard.is_paused:
            return

        t_now = time.time()
        config = dashboard.get_current_config()
        loss_breakdown = dashboard.trainer.step(config)

        # Rate-limited UI statistics (~3 Hz)
        if t_now - last_ui_update >= 0.3:
            train_times = dashboard.trainer.train_times
            it_s = 1.0 / max(sum(train_times) / len(train_times), 1e-4) if train_times else 0.0

            dashboard.stats_label.value = f"""
            <div style='font-family: monospace; font-size: 11px; color: #FFD700; background: #1a1a1a; padding: 10px; border-radius: 4px; border: 1px solid #333; line-height: 1.4; width: 230px;'>
              <div style='font-weight: bold; color: #FFF; margin-bottom: 5px; font-size: 11px;'>⚡ REAL-TIME METRICS ({it_s:.0f} it/s)</div>
              <div>🏆 Best Feature Loss: <span style='color: #00FF00; font-weight: bold;'>{dashboard.trainer.best_loss:.4f}</span></div>
              <div style='color: #444; margin: 4px 0;'>---------------------------------------------</div>
              <div>  • Total Loss:               <span style='color: #1f77b4; font-weight: bold;'>{loss_breakdown['total']:.4f}</span></div>
              <div>  • Feature/Dream:             <span style='color: #ff7f0e; font-weight: bold;'>{loss_breakdown['feature']:.4f}</span></div>
              <div>  • L2 Reg Loss:              <span style='color: #2ca02c; font-weight: bold;'>{loss_breakdown['l2']:.4f}</span></div>
              <div>  • TV Reg Loss:              <span style='color: #d62728; font-weight: bold;'>{loss_breakdown['tv']:.4f}</span></div>
              <div>  • Spectral Crest:           <span style='color: #9467bd; font-weight: bold;'>{loss_breakdown['sparsity']:.4f}</span></div>
            </div>
            """
            last_ui_update = t_now

        # Rate-limited training plot refresh (~0.3 Hz)
        logs = dashboard.trainer.logs
        if len(logs["total"]) > 0 and (t_now - last_graph_update >= 3.0):
            # Apply dark theme styling to match UI
            plt.style.use('dark_background')
            fig, ax = plt.subplots(figsize=(6, 2.5), facecolor='#1a1a1a')
            ax.set_facecolor('#1a1a1a')

            ax.plot(logs["total"][-500:], color='#1f77b4', linewidth=1.5, label='Total Loss')
            ax.plot(logs["feature"][-500:], color='#ff7f0e', linewidth=1.2, linestyle='--', label='Feature/Dream')
            ax.plot(logs["l2"][-500:], color='#2ca02c', linewidth=1.0, alpha=0.7, label='L2 Reg')
            ax.plot(logs["tv"][-500:], color='#d62728', linewidth=1.0, alpha=0.7, label='TV Reg')
            ax.plot(logs["sparsity"][-500:], color='#9467bd', linewidth=1.0, alpha=0.7, linestyle=':', label='Spectral Sparsity')
            ax.legend(loc='upper right', fontsize=8, framealpha=0.3)

            title_tag = f"Targeting {config['target_layer']}"
            if config['is_deep_dream']:
                title_tag += " (DeepDream)"
            else:
                title_tag += f" C{config['target_channel']}"

            ax.set_title(title_tag, fontsize=10, color='#ccc')
            ax.grid(True, color='#333', linestyle='--', alpha=0.5)
            ax.tick_params(colors='#aaa', labelsize=8)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('#444')
            ax.spines['bottom'].set_color('#444')
            fig.tight_layout(pad=1.0)

            buf = io.BytesIO()
            fig.savefig(buf, format='png', bbox_inches='tight', facecolor='#1a1a1a')
            plt.close(fig)
            dashboard.graph_widget.value = buf.getvalue()
            last_graph_update = t_now
    except Exception:
         print("\nTRAIN ERROR:\n", traceback.format_exc())


def colab_perturb(x_norm, y_norm):
    try:
        h, w = 128, 128
        grid_y = int(y_norm * h)

        if x_norm < 0.33:
            grid_x = int((x_norm * 3.0) * w)
            Y, X = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing='ij')
            dist = (X - grid_x)**2 + (Y - grid_y)**2
            mask = (dist > 15**2).float().view(1, 1, h, w).to(device)
            dashboard.display_x0 = dashboard.display_x0 * mask
        elif x_norm < 0.66:
            grid_x = int(((x_norm - 0.33) * 3.0) * w)
            Y, X = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing='ij')
            dist = (X - grid_x)**2 + (Y - grid_y)**2
            mask = (dist > 15**2).float().view(1, 1, h, w).to(device)

            # Map frequency domain back to spatial coordinate frame, apply mask, and return
            z_spatial = torch.fft.ifft2(torch.fft.ifftshift(dashboard.z_rk4))
            z_spatial = z_spatial * mask
            dashboard.z_rk4 = torch.fft.fftshift(torch.fft.fft2(z_spatial))
    except Exception:
        print("\nPerturbation Error:\n", traceback.format_exc())


# Register Colab callbacks
output.register_callback('notebook.colab_render_step', colab_render_step)
output.register_callback('notebook.colab_train_step', colab_train_step)
output.register_callback('notebook.colab_perturb', colab_perturb)

# Javasript pipeline loop controller
display(HTML("""
<style>
.colab-nca-viewport img {
    cursor: crosshair !important;
    border: 1px solid #444;
    border-radius: 2px;
    image-rendering: pixelated !important;
    image-rendering: crisp-edges !important;
}
</style>

<script>
(async function() {
    if (window.colab_render_interval) {
        clearInterval(window.colab_render_interval);
    }
    window.colab_train_active = false;
    await new Promise(resolve => setTimeout(resolve, 150));

    const viewportContainer = document.querySelector('.colab-nca-viewport');
    if (viewportContainer) {
        const img = viewportContainer.querySelector('img');
        if (img) {
            let isDrawing = false;

            const handlePaint = (e) => {
                const rect = img.getBoundingClientRect();
                const x = (e.clientX - rect.left) / rect.width;
                const y = (e.clientY - rect.top) / rect.height;
                if (x >= 0 && x <= 1 && y >= 0 && y <= 1) {
                    google.colab.kernel.invokeFunction('notebook.colab_perturb', [x, y], {});
                }
            };

            img.addEventListener('mousedown', (e) => {
                isDrawing = true;
                handlePaint(e);
            });

            img.addEventListener('mousemove', (e) => {
                if (isDrawing) {
                    handlePaint(e);
                }
            });

            window.addEventListener('mouseup', () => {
                isDrawing = false;
            });
            console.log("Interactives initialized.");
        }
    }

    window.colab_train_active = true;
    window.colab_render_interval = setInterval(() => {
        google.colab.kernel.invokeFunction('notebook.colab_render_step', [], {});
    }, 33);

    while (window.colab_train_active) {
        try {
            await google.colab.kernel.invokeFunction('notebook.colab_train_step', [], {});
        } catch (e) {
            console.error("Train pipeline error:", e);
            await new Promise(resolve => setTimeout(resolve, 1000));
        }
        await new Promise(resolve => setTimeout(resolve, 2));
    }
})();
</script>
"""))

In [2]:
1

1